# 🌍 Intelligent Environmental Data Analytics System
### Climate Pattern Analysis Using Machine Learning — Training Notebook

This notebook:
1. Generates a realistic synthetic environmental dataset (10 years × 6 regions)
2. Engineers a 5-class **Climate Risk Category** label
3. Trains a **Random Forest Classifier**
4. Fits a **K-Means** model for unsupervised pattern discovery
5. Saves everything into `model/climate_model.pkl`
6. Downloads `climate_model.pkl` and `environmental_data.csv` so you can drop them
   straight into your Streamlit project's `model/` and `data/` folders.

> Run all cells top to bottom: **Runtime → Run all**


## 1️⃣ Install & Import Dependencies

In [ ]:
!pip install -q scikit-learn pandas numpy joblib


In [ ]:
import numpy as np
import pandas as pd
import joblib
import os

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.cluster import KMeans

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

os.makedirs("model", exist_ok=True)
os.makedirs("data", exist_ok=True)
print("Setup complete ✅")


## 2️⃣ Generate Synthetic Environmental Dataset\n\n6 regions × 10 years × 12 months of realistic sensor readings (temperature, humidity, rainfall, CO₂, wind speed, pressure) with seasonal cycles and a long-term warming/CO₂ trend.

In [ ]:
REGIONS = {
    # region: (base_temp, base_humidity, base_rainfall, base_co2)
    "Coastal Plains":     (26.0, 75, 120, 415),
    "Highland Plateau":   (14.0, 55, 80,  405),
    "Arid Basin":         (33.0, 25, 15,  420),
    "Tropical Belt":      (29.0, 85, 220, 418),
    "Temperate Valley":   (18.0, 60, 95,  410),
    "Urban Metro":        (24.0, 50, 70,  435),
}

YEARS = list(range(2015, 2025))
MONTHS = list(range(1, 13))

records = []
for region, (base_t, base_h, base_r, base_c) in REGIONS.items():
    for year in YEARS:
        year_idx = year - YEARS[0]
        warming_trend = year_idx * rng.uniform(0.03, 0.07)
        co2_trend = year_idx * rng.uniform(1.2, 2.0)

        for month in MONTHS:
            seasonal = np.sin((month - 1) / 12 * 2 * np.pi)

            temperature = base_t + warming_trend + seasonal * 6 + rng.normal(0, 1.8)
            humidity = np.clip(base_h - seasonal * 10 + rng.normal(0, 6), 5, 100)
            rainfall = np.clip(base_r * (1 + 0.5 * seasonal) + rng.normal(0, 20), 0, None)
            co2_level = base_c + co2_trend + rng.normal(0, 2.5)
            wind_speed = np.clip(8 + rng.normal(0, 3.5) + (2 if region == "Coastal Plains" else 0), 0, None)
            pressure = np.clip(1013 - seasonal * 4 + rng.normal(0, 3), 970, 1040)

            records.append({
                "Region": region, "Year": year, "Month": month,
                "Temperature_C": round(temperature, 2),
                "Humidity_pct": round(humidity, 2),
                "Rainfall_mm": round(rainfall, 2),
                "CO2_ppm": round(co2_level, 2),
                "Wind_Speed_kmh": round(wind_speed, 2),
                "Pressure_hPa": round(pressure, 2),
            })

df = pd.DataFrame(records)
print(f"Generated {len(df):,} records")
df.head()


## 3️⃣ Engineer the Climate Risk Category Label\n\nRule-based scoring (with stochastic noise so it's a genuine learning problem, not a lookup table) across 5 classes: `Normal`, `Heatwave Risk`, `Drought Risk`, `Flood Risk`, `Extreme Weather`.

In [ ]:
def assign_risk(row):
    score = {
        "Normal": 1.0,
        "Heatwave Risk": 0.0,
        "Drought Risk": 0.0,
        "Flood Risk": 0.0,
        "Extreme Weather": 0.0,
    }

    if row.Temperature_C > 34 and row.Humidity_pct < 40:
        score["Heatwave Risk"] += 3.0
    elif row.Temperature_C > 30:
        score["Heatwave Risk"] += 1.2

    if row.Rainfall_mm < 30 and row.Humidity_pct < 35:
        score["Drought Risk"] += 3.0
    elif row.Rainfall_mm < 50:
        score["Drought Risk"] += 1.0

    if row.Rainfall_mm > 220 and row.Humidity_pct > 80:
        score["Flood Risk"] += 3.0
    elif row.Rainfall_mm > 170:
        score["Flood Risk"] += 1.2

    if row.Wind_Speed_kmh > 16 or row.Pressure_hPa < 990:
        score["Extreme Weather"] += 2.5
    elif row.Wind_Speed_kmh > 12:
        score["Extreme Weather"] += 1.0

    score["Normal"] = 1.0

    noise = rng.normal(0, 0.35, size=len(score))
    labels = list(score.keys())
    values = np.array(list(score.values())) + noise
    return labels[int(np.argmax(values))]

df["Climate_Risk_Category"] = df.apply(assign_risk, axis=1)

print("Class distribution:")
print(df["Climate_Risk_Category"].value_counts())

df.to_csv("data/environmental_data.csv", index=False)
print("\nSaved -> data/environmental_data.csv")


## 4️⃣ Prepare Features

In [ ]:
FEATURE_COLS = [
    "Temperature_C", "Humidity_pct", "Rainfall_mm",
    "CO2_ppm", "Wind_Speed_kmh", "Pressure_hPa",
]

region_encoder = LabelEncoder()
df["Region_encoded"] = region_encoder.fit_transform(df["Region"])

X = df[FEATURE_COLS + ["Region_encoded"]].copy()
y = df["Climate_Risk_Category"]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=RANDOM_STATE, stratify=y_encoded
)

print(f"Train shape: {X_train.shape}  |  Test shape: {X_test.shape}")


## 5️⃣ Train the Random Forest Classifier

In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="weighted")

print(f"Test Accuracy: {acc:.4f}")
print(f"Weighted F1:   {f1:.4f}\n")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

feature_importance = dict(zip(FEATURE_COLS + ["Region_encoded"], model.feature_importances_))


## 6️⃣ Unsupervised Pattern Discovery (K-Means)

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=10)
kmeans.fit(X_scaled)
print("KMeans fitted with 4 clusters ✅")


## 7️⃣ Bundle & Save the Model Pickle

In [ ]:
bundle = {
    "model": model,
    "scaler": scaler,
    "label_encoder": label_encoder,
    "region_encoder": region_encoder,
    "kmeans": kmeans,
    "feature_cols": FEATURE_COLS,
    "class_names": list(label_encoder.classes_),
    "regions": list(region_encoder.classes_),
    "feature_importance": feature_importance,
    "metrics": {"accuracy": acc, "f1_weighted": f1},
}

joblib.dump(bundle, "model/climate_model.pkl")
print("Saved -> model/climate_model.pkl")
print("Saved -> data/environmental_data.csv")


## 8️⃣ Download the Files (for your Streamlit project)\n\nThis downloads `climate_model.pkl` and `environmental_data.csv` to your computer. Place them at:\n```\nclimate_ml_project/model/climate_model.pkl\nclimate_ml_project/data/environmental_data.csv\n```

In [ ]:
try:
    from google.colab import files
    files.download("model/climate_model.pkl")
    files.download("data/environmental_data.csv")
except ImportError:
    print("Not running in Google Colab — find the files in the 'model/' and 'data/' folders.")


## 9️⃣ (Optional) Save Directly to Google Drive\n\nUncomment and run this cell instead if you'd rather save straight to Drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
#
# import shutil
# drive_path = "/content/drive/MyDrive/climate_ml_project"
# os.makedirs(f"{drive_path}/model", exist_ok=True)
# os.makedirs(f"{drive_path}/data", exist_ok=True)
# shutil.copy("model/climate_model.pkl", f"{drive_path}/model/climate_model.pkl")
# shutil.copy("data/environmental_data.csv", f"{drive_path}/data/environmental_data.csv")
# print("Saved to Google Drive ✅")
